In [ ]:
import pandas as pd
import numpy as np
import random
from datetime import datetime, timedelta

# ==============================================================================
# 1. ATURAN & POLA TAMBAHAN (True Positive = 1)
# Format: (rule.description, rule.level, rule.id, data.alert.signature_id)
# ==============================================================================
tp_rules = [
    # DDoS & Flood
    ("CUSTOM Possible TCP SYN Flood", 10, "100001", "2010001"),
    ("CUSTOM TEST TCP SYN", 8, "100002", "2010002"),
    ("SYN flood detected", 9, "100003", "2010003"),
    ("UDP Flood attack against internal server", 8, "100004", "2010004"),

    # Brute Force & Credential Attacks
    ("SSH Brute Force", 10, "5712", "2010010"),
    ("sshd: brute force", 10, "5712", "2010011"),
    ("PAM: Multiple failed logins", 9, "5503", "2010020"),
    ("Maximum authentication attempts exceeded", 9, "5713", "2010021"),
    ("Possible Mimikatz command execution", 12, "92001", "2010022"),
    ("Kerberoasting attack detected", 11, "92002", "2010023"),

    # Reconnaissance & Scanning
    ("Port Scan", 7, "100010", "2010030"),
    ("ET SCAN Potential SSH Scan", 7, "2001219", "2001219"),
    ("ET SCAN SFTP/FTP Password Exposure", 7, "2001220", "2001220"),
    ("ET SCAN NMAP OS Detection Probe", 7, "2000537", "2000537"),
    ("Nikto Web App Vulnerability Scan", 8, "31105", "2010035"),
    ("Masscan fast port scan detected", 8, "100012", "2010036"),

    # Web Exploits & Injection
    ("SQL Injection", 12, "31101", "2010040"),
    ("XSS Attempt", 11, "31102", "2010050"),
    ("Remote Code Execution attempt via HTTP", 12, "31103", "2010051"),
    ("Directory Traversal attack detected", 10, "31104", "2010052"),
    ("Command Injection via web parameter", 12, "31106", "2010053"),
    ("ET INFO Request to Hidden Environment File", 6, "2024364", "2024364"),
    ("ET WEB_SERVER Tilde in URI", 6, "2003492", "2003492"),
    ("GPL WEB_SERVER global.asa access", 8, "2101449", "2101449"),
    ("GPL WEB_SERVER .htaccess access", 8, "2101448", "2101448"),
    ("GPL WEB_SERVER .htpasswd access", 8, "2101447", "2101447"),
    ("ET WEB_SERVER WEB-PHP phpinfo access", 7, "2008581", "2008581"),

    # Malware, Reverse Shell & Post-Exploitation
    ("Reverse shell connection established", 13, "100200", "2010060"),
    ("Meterpreter payload execution", 13, "100201", "2010061"),
    ("Cobalt Strike Beacon activity detected", 14, "100202", "2010062"),
    ("Ransomware file extension modification pattern", 14, "100203", "2010063"),
    ("Crypto miner background process spawned", 11, "100204", "2010064"),

    # Threat Intelligence, C2 & Data Exfiltration
    ("ET CINS Active Threat Intelligence Poor Reputation IP", 9, "2022879", "2022879"),
    ("ET DROP Dshield", 8, "2402000", "2402000"),
    ("ET DROP Spamhaus", 8, "2403339", "2403339"),
    ("ET TOR Known Tor Relay Traffic", 8, "2404000", "2404000"),
    ("ET HUNTING ZIP file exfiltration", 10, "2031111", "2031111"),
    ("DNS Tunneling query detected", 10, "2031112", "2031112"),
    ("SURICATA STREAM 3way handshake SYN resend different seq", 7, "2210044", "2210044")
]

# ==============================================================================
# 2. ATURAN & POLA TAMBAHAN (False Positive = 0)
# Format: (rule.description, rule.level, rule.id, data.alert.signature_id)
# ==============================================================================
fp_rules = [
    # SSH & Autentikasi Normal
    ("SSH Login", 3, "5715", "1001"),
    ("Successful SSH Login", 3, "5715", "1002"),
    ("PAM: User login failed.", 5, "5501", "1003"),
    ("PAM: Login session opened", 3, "5501", "1004"),
    ("PAM: Login session closed", 3, "5502", "1005"),
    ("sshd: authentication failed.", 5, "5716", "1006"),
    ("sshd: authentication success.", 3, "5715", "1007"),
    ("sshd: Attempt to login using a non-existent user", 5, "5710", "1008"),
    ("sshd: session opened", 3, "5715", "1009"),

    # HTTP & Web Noise Normal
    ("Nginx Access", 3, "30101", "1010"),
    ("HTTP Request", 3, "30102", "1011"),
    ("DNS Query", 2, "30103", "1012"),
    ("Nginx 404 Not Found response", 4, "30104", "1013"),
    ("Nginx 200 OK access log", 2, "30105", "1014"),
    ("Apache access log request", 3, "30106", "1015"),

    # Package Manager & System Update
    ("Package Update", 3, "2901", "1020"),
    ("APT Update", 3, "2902", "1021"),
    ("New dpkg (Debian Package)", 3, "2903", "1022"),
    ("Dpkg (Debian Package) half", 3, "2904", "1023"),
    ("System update completed", 3, "2905", "1024"),

    # Wazuh & Agent Telemetry
    ("Wazuh agent started", 3, "501", "1030"),
    ("Wazuh agent stopped", 3, "502", "1031"),
    ("Agent event queue is 90% full", 4, "503", "1032"),
    ("Agent event queue is back to normal load", 3, "504", "1033"),
    ("Wazuh agent disconnected", 3, "505", "1034"),
    ("Wazuh agent connected", 3, "506", "1035"),

    # Rootcheck & File Integrity Monitoring (FIM)
    ("Host-based anomaly detection event (rootcheck).", 3, "510", "1040"),
    ("Integrity checksum changed for system file", 3, "550", "1041"),
    ("File added to the system", 3, "554", "1042"),
    ("File modified during software installation", 3, "551", "1043"),

    # System Admin, Cron & Service Logs
    ("Successful sudo to ROOT executed", 3, "5402", "1050"),
    ("First time user executed sudo", 4, "5401", "1051"),
    ("New user added to the system", 5, "5901", "1052"),
    ("New group added to the system", 5, "5902", "1053"),
    ("Group (or user) deleted from the system", 5, "5903", "1054"),
    ("crontab job executed scheduled backup", 3, "5301", "1055"),
    ("Systemd service started unit", 3, "5302", "1056"),

    # Auditd & Kernel Syscalls
    ("Auditd: Device enables promiscuous mode", 4, "80700", "1060"),
    ("Auditd: SELinux permission check", 3, "80701", "1061"),
    ("Auditd: Process executed /bin/ls", 3, "80702", "1062"),

    # Suricata Benign Network Events
    ("SURICATA STREAM 3way handshake SYNACK resend with different ack", 4, "2210045", "2210045"),
    ("SURICATA Applayer Mismatch protocol both directions", 4, "2221010", "2221010"),
    ("ET INFO Go-http-client User-Agent Observed Inbound", 3, "2013028", "2013028"),
    ("SURICATA STREAM packet with invalid ack", 3, "2210046", "2210046"),
    ("SURICATA STREAM ESTABLISHED retransmission", 3, "2210047", "2210047")
]

# ==============================================================================
# 3. GENERASI 2.000 SINTETIS DATA (1396 FP : 604 TP)
# ==============================================================================
n_fp = 1396
n_tp = 604
rows = []
base_time = datetime(2026, 3, 1, 8, 0, 0)

# Tambahkan sampel False Positive (label = 0)
for _ in range(n_fp):
    desc, level, rule_id, sig_id = random.choice(fp_rules)
    dt = base_time + timedelta(minutes=random.randint(0, 43200), seconds=random.randint(0, 59))
    ts_str = dt.strftime("%b %d, %Y @ %H:%M:%S.%f")[:-3]
    severity = float(level) if random.random() > 0.08 else np.nan
    rows.append({
        "timestamp": ts_str,
        "rule.level": level,
        "rule.id": rule_id,
        "rule.description": desc,
        "data.alert.severity": severity,
        "data.alert.signature_id": sig_id,
        "agent.name": random.choice(["agent-vm1", "soar-vm", "wazuh-agent-linux", "srv-db01"]),
        "label": 0
    })

# Tambahkan sampel True Positive (label = 1)
for _ in range(n_tp):
    desc, level, rule_id, sig_id = random.choice(tp_rules)
    dt = base_time + timedelta(minutes=random.randint(0, 43200), seconds=random.randint(0, 59))
    ts_str = dt.strftime("%b %d, %Y @ %H:%M:%S.%f")[:-3]
    severity = float(level) if random.random() > 0.08 else np.nan
    rows.append({
        "timestamp": ts_str,
        "rule.level": level,
        "rule.id": rule_id,
        "rule.description": desc,
        "data.alert.severity": severity,
        "data.alert.signature_id": sig_id,
        "agent.name": random.choice(["agent-vm1", "soar-vm", "wazuh-agent-linux", "srv-db01"]),
        "label": 1
    })

# Acak baris dan simpan ke CSV
df_synthetic = pd.DataFrame(rows).sample(frac=1, random_state=42).reset_index(drop=True)
df_synthetic.to_csv("wazuh_alerts_labeled(2).csv", index=False)

print("✅ Dataset berhasil diperluas dan disimpan: wazuh_alerts_labeled(2).csv")
print(f"Total baris: {len(df_synthetic)}")
print(df_synthetic["label"].value_counts())

✅ Dataset berhasil diperluas dan disimpan: wazuh_alerts_labeled(2).csv
Total baris: 2000
label
0    1396
1     604
Name: count, dtype: int64


In [ ]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report, confusion_matrix
import warnings
warnings.filterwarnings('ignore')
from sklearn.metrics import make_scorer, average_precision_score
from imblearn.pipeline import Pipeline as ImbPipeline
from imblearn.over_sampling import SMOTE
from sklearn import pipeline

In [ ]:
df = pd.read_csv("/content/wazuh_alerts_labeled(2).csv")

In [ ]:
# 2. Handle Timestamp (Convert string to datetime features)
df['timestamp'] = pd.to_datetime(df['timestamp'], format='%b %d, %Y @ %H:%M:%S.%f')
df['hour'] = df['timestamp'].dt.hour
df['day_of_week'] = df['timestamp'].dt.dayofweek
df['is_weekend'] = df['day_of_week'].isin([5, 6]).astype(int)

# 3. Drop useless columns
df.drop(columns=['timestamp', 'agent.name', 'rule.description'], inplace=True)

# 4. Encode Categorical Features (rule.id, signature_id)
# Option A: Label Encoding (Good for Tree-based models like XGBoost/RandomForest)
from sklearn.preprocessing import LabelEncoder

le_id = LabelEncoder()
df['rule.id'] = le_id.fit_transform(df['rule.id'].astype(str))

le_sig = LabelEncoder()
df['data.alert.signature_id'] = le_sig.fit_transform(df['data.alert.signature_id'].astype(str))

# If data.alert.severity is missing, use rule.level instead
df['data.alert.severity'] = df['data.alert.severity'].fillna(df['rule.level'])

# 5. Final Feature Selection
features = ['rule.level', 'rule.id', 'data.alert.severity',
            'data.alert.signature_id', 'hour', 'day_of_week', 'is_weekend']
target = 'label'

X = df[features]
y = df[target]

print("✅ Data ready for modeling!")
print(X.head())

✅ Data ready for modeling!
   rule.level  rule.id  data.alert.severity  data.alert.signature_id  hour  \
0          10       20                 10.0                       72    12   
1           3       75                  3.0                       38     8   
2           5       72                  5.0                       33    16   
3           2       41                  2.0                       13     6   
4           3       49                  3.0                       20    14   

   day_of_week  is_weekend  
0            3           0  
1            2           0  
2            1           0  
3            1           0  
4            3           0  


In [ ]:
print("Distribusi Label:")
print(df['label'].value_counts())
print(f"\nRasio Ketimpangan: {(df['label']==0).sum() / max((df['label']==1).sum(), 1):.1f}:1")

Distribusi Label:
label
0    1396
1     604
Name: count, dtype: int64

Rasio Ketimpangan: 2.3:1


In [ ]:
cat_cols = ['rule.id', 'data.alert.signature_id']
num_cols = ['rule.level', 'data.alert.severity', 'hour', 'day_of_week', 'is_weekend']

In [ ]:
print(df['label'].value_counts())

import sklearn
print(sklearn.__version__)

label
0    1396
1     604
Name: count, dtype: int64
1.6.1


In [ ]:
from sklearn.model_selection import GridSearchCV, StratifiedKFold
from sklearn.metrics import make_scorer, average_precision_score, f1_score
import numpy as np
import warnings
warnings.filterwarnings('ignore')
from sklearn.pipeline import Pipeline # Import Pipeline
from sklearn.ensemble import RandomForestClassifier

def safe_avg_precision(y_true, y_pred_proba):
    try:
        return average_precision_score(y_true, y_pred_proba)
    except ValueError:
        return 0.0

# Parameter grid TERBATAS (fokus pada sweet spot)
param_grid = {
    'rf__max_depth': [12, 15],           # Naikkan sedikit dari 10
    'rf__min_samples_leaf': [10, 15],     # Turunkan sedikit dari 20
    'rf__class_weight': [{0:1, 1:3}, {0:1, 1:4}],  # Coba weight lebih agresif
    'rf__n_estimators': [300, 500]       # Tetap tinggi untuk stabilitas
}

pipe = Pipeline([('rf', RandomForestClassifier(
    min_samples_split=30,
    max_features='log2',
    max_samples=0.75,
    random_state=42,
    n_jobs=-1
))])

cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

grid = GridSearchCV(
    pipe, param_grid,
    cv=cv,
    scoring=make_scorer(safe_avg_precision, response_method='predict_proba'),  # Optimize AP langsung!
    n_jobs=-1,
    refit=True
)

grid.fit(df[num_cols + cat_cols], df[target])

print(f" Best Params: {grid.best_params_}")
print(f" Best CV Avg Precision: {grid.best_score_:.4f}")

# Evaluasi final dengan best estimator
best_pipe = grid.best_estimator_
results_final = cross_validate(
    best_pipe, df[num_cols + cat_cols], df[target],
    cv=cv,
    scoring={
        'accuracy': 'accuracy',
        'f1_weighted': 'f1_weighted',
        'avg_precision': make_scorer(safe_avg_precision, response_method='predict_proba'),
        'recall_macro': 'recall_macro'
    },
    return_train_score=True
)

print("\n" + "="*60)
print(" FINAL EVALUATION")
print("="*60)
for metric in ['accuracy', 'f1_weighted', 'avg_precision', 'recall_macro']:
    train_mean = results_final[f'train_{metric}'].mean()
    test_mean = results_final[f'test_{metric}'].mean()
    gap = train_mean - test_mean
    status = "⚠️ OVERFITTING" if gap > 0.05 else "✅ OK"
    print(f"{metric:>15}: Train={train_mean:.4f} | Test={test_mean:.4f} | Gap={gap:.4f} {status}")

 Best Params: {'rf__class_weight': {0: 1, 1: 3}, 'rf__max_depth': 12, 'rf__min_samples_leaf': 10, 'rf__n_estimators': 300}
 Best CV Avg Precision: nan

 FINAL EVALUATION
       accuracy: Train=1.0000 | Test=1.0000 | Gap=0.0000 ✅ OK
    f1_weighted: Train=1.0000 | Test=1.0000 | Gap=0.0000 ✅ OK
  avg_precision: Train=nan | Test=nan | Gap=nan ✅ OK
   recall_macro: Train=1.0000 | Test=1.0000 | Gap=0.0000 ✅ OK


In [ ]:
from sklearn.pipeline import Pipeline  # Pakai pipeline biasa, bukan ImbPipeline
from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.metrics import make_scorer, average_precision_score
import warnings
warnings.filterwarnings('ignore')

# ============================================================
# 1. HANYA PAKAI CLASS_WEIGHT (Cukup untuk rasio 2.3:1)
# ============================================================
rf_params = {
    'n_estimators': 300,
    'max_depth': 15,
    'min_samples_split': 20,
    'min_samples_leaf': 10,
    'max_features': 'sqrt',
    'max_samples': 0.8,
    'class_weight': 'balanced',       # ← INI SAJA SUDAH CUKUP!
    'random_state': 42,
    'n_jobs': -1
}

# Pakai Pipeline SKLEARN BIASA (tanpa SMOTE)
pipe = Pipeline([
    ('rf', RandomForestClassifier(**rf_params))
])

# ============================================================
# 2. CROSS-VALIDATION
# ============================================================
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

scoring = {
    'accuracy': 'accuracy',
    'f1_weighted': 'f1_weighted',
    'avg_precision': make_scorer(average_precision_score, response_method='predict_proba'),
    'recall_macro': 'recall_macro'
}

results = cross_validate(
    pipe,
    df[num_cols + cat_cols],
    df[target],
    cv=cv,
    scoring=scoring,
    return_train_score=True
)

# ============================================================
# 3. HASIL
# ============================================================
print("\n" + "="*60)
print("🔍 OVERFITTING CHECK")
print("="*60)

for metric in ['accuracy', 'f1_weighted', 'avg_precision', 'recall_macro']:
    train_mean = results[f'train_{metric}'].mean()
    test_mean = results[f'test_{metric}'].mean()
    gap = train_mean - test_mean
    status = "⚠️ OVERFITTING" if gap > 0.05 else "✅ OK"
    print(f"{metric:>15}: Train={train_mean:.4f} | Test={test_mean:.4f} | Gap={gap:.4f} {status}")

print(f"\n Final Test Scores (Mean ± Std):")
for metric in ['accuracy', 'f1_weighted', 'avg_precision', 'recall_macro']:
    print(f"  {metric:>15}: {results[f'test_{metric}'].mean():.4f} ± {results[f'test_{metric}'].std():.4f}")


🔍 OVERFITTING CHECK
       accuracy: Train=1.0000 | Test=1.0000 | Gap=0.0000 ✅ OK
    f1_weighted: Train=1.0000 | Test=1.0000 | Gap=0.0000 ✅ OK
  avg_precision: Train=nan | Test=nan | Gap=nan ✅ OK
   recall_macro: Train=1.0000 | Test=1.0000 | Gap=0.0000 ✅ OK

 Final Test Scores (Mean ± Std):
         accuracy: 1.0000 ± 0.0000
      f1_weighted: 1.0000 ± 0.0000
    avg_precision: nan ± nan
     recall_macro: 1.0000 ± 0.0000


In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import precision_recall_curve, f1_score
import numpy as np

# ============================================================
# 1. SPLIT DATA SECARA MANUAL (Hanya untuk Threshold Tuning)
# ============================================================
X = df[num_cols + cat_cols]
y = df[target]

# Bagi data 80% train, 20% test (stratified agar rasio kelas tetap)
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y          # ← PENTING: Jaga distribusi kelas minoritas
)

print(f"Train size: {len(X_train)} | Test size: {len(X_test)}")
print(f"Test class dist: {y_test.value_counts().to_dict()}")

# ============================================================
# 2. FIT MODEL & DAPATKAN PROBABILITAS
# ============================================================
pipe.fit(X_train, y_train)

# Ambil probabilitas kelas positif (label=1)
y_proba = pipe.predict_proba(X_test)[:, 1]

# ============================================================
# 3. CARI THRESHOLD TERBAIK BERDASARKAN F1-SCORE
# ============================================================
precisions, recalls, thresholds = precision_recall_curve(y_test, y_proba)

# Hitung F1-Score untuk setiap threshold
f1_scores = 2 * (precisions * recalls) / (precisions + recalls + 1e-8)
best_idx = np.argmax(f1_scores)
best_threshold = thresholds[best_idx] if best_idx < len(thresholds) else 0.5

print(f"\n🎯 Best Threshold: {best_threshold:.3f}")
print(f"   F1 at best threshold: {f1_scores[best_idx]:.4f}")
print(f"   Precision at best: {precisions[best_idx]:.4f}")
print(f"   Recall at best: {recalls[best_idx]:.4f}")

# ============================================================
# 4. BANDINGKAN DENGAN DEFAULT THRESHOLD (0.5)
# ============================================================
y_pred_default = (y_proba >= 0.5).astype(int)
y_pred_best = (y_proba >= best_threshold).astype(int)

print(f"\n📊 Comparison:")
print(f"  Default (0.5) → F1: {f1_score(y_test, y_pred_default):.4f}")
print(f"  Best ({best_threshold:.3f}) → F1: {f1_score(y_test, y_pred_best):.4f}")

Train size: 1600 | Test size: 400
Test class dist: {0: 279, 1: 121}

🎯 Best Threshold: 0.986
   F1 at best threshold: 1.0000
   Precision at best: 1.0000
   Recall at best: 1.0000

📊 Comparison:
  Default (0.5) → F1: 1.0000
  Best (0.986) → F1: 1.0000


In [ ]:
'''
RULE_COLUMN = "rule.description"

# =====================================================
# Labeling Rules
# 1 = True Positive
# 0 = False Positive
# -1 = Unknown (Need Manual Review)
# =====================================================

label_rules = {

    # ==================================================
    # TRUE POSITIVE (1)
    # ==================================================

    # DDoS
    "CUSTOM Possible TCP SYN Flood": 1,
    "CUSTOM TEST TCP SYN": 1,

    # Brute Force
    "SSH Brute Force": 1,
    "sshd: brute force": 1,
    "PAM: Multiple failed logins": 1,

    # Reconnaissance
    "Port Scan": 1,
    "ET SCAN Potential SSH Scan": 1,
    "ET SCAN SFTP/FTP Password Exposure": 1,

    # Web Attack
    "SQL Injection": 1,
    "XSS Attempt": 1,
    "ET INFO Request to Hidden Environment File": 1,
    "ET WEB_SERVER Tilde in URI": 1,
    "GPL WEB_SERVER global.asa access": 1,
    "GPL WEB_SERVER .htaccess access": 1,
    "GPL WEB_SERVER .htpasswd access": 1,
    "ET WEB_SERVER WEB-PHP phpinfo access": 1,

    # Threat Intelligence
    "ET CINS Active Threat Intelligence Poor Reputation IP": 1,
    "ET DROP Dshield": 1,
    "ET DROP Spamhaus": 1,

    # Data Exfiltration
    "ET HUNTING ZIP file exfiltration": 1,

    # Suspicious TCP Behaviour
    "SURICATA STREAM 3way handshake SYN resend different seq": 1,



    # ==================================================
    # FALSE POSITIVE (0)
    # ==================================================

    # SSH / Login
    "SSH Login": 0,
    "Successful SSH Login": 0,
    "PAM: User login failed.": 0,
    "PAM: Login session opened": 0,
    "PAM: Login session closed": 0,
    "sshd: authentication failed.": 0,
    "sshd: authentication success.": 0,
    "sshd: Attempt to login using a non-existent user": 0,

    # HTTP / DNS
    "Nginx Access": 0,
    "HTTP Request": 0,
    "DNS Query": 0,

    # Package Management
    "Package Update": 0,
    "APT Update": 0,
    "New dpkg (Debian Package)": 0,
    "Dpkg (Debian Package) half": 0,

    # Wazuh
    "Wazuh agent started": 0,
    "Wazuh agent stopped": 0,
    "Agent event queue is 90% full": 0,
    "Agent event queue is back to normal load": 0,

    # Rootcheck
    "Host-based anomaly detection event (rootcheck).": 0,

    # System Administration
    "Successful sudo to ROOT executed": 0,
    "First time user executed sudo": 0,
    "New user added to the system": 0,
    "New group added to the system": 0,
    "Group (or user) deleted from the system": 0,

    # Auditd
    "Auditd: Device enables promiscuous mode": 0,
    "Auditd: SELinux permission check": 0,

    # Suricata Engine Events
    "SURICATA STREAM 3way handshake SYNACK resend with different ack": 0,
    "SURICATA Applayer Mismatch protocol both directions": 0,
    "ET INFO Go-http-client User-Agent Observed Inbound": 0,
}

def auto_label(description):

    if pd.isna(description):
        return -1

    description = str(description)

    for keyword, label in label_rules.items():
        if keyword.lower() in description.lower():
            return label

    return -1


df["label"] = df[RULE_COLUMN].apply(auto_label)

df.to_csv("wazuh_alerts_labeled(2).csv", index=False)

print("Done!")
print(df["label"].value_counts())
'''

In [ ]:
'''
unknown = df[df["label"] == -1]

print(
    unknown["rule.description"]
    .value_counts()
)
'''

In [ ]:
import joblib
import json

# Export model (pipe sudah fit di cell sebelumnya)
joblib.dump(pipe, 'wazuh_fp_model.pkl')

# Export LabelEncoders (penting! untuk encode rule.id & signature_id)
joblib.dump(le_id, 'le_rule_id.pkl')
joblib.dump(le_sig, 'le_signature_id.pkl')

# Export config
config = {
    "threshold": 0.491,
    "features": ["rule.level", "data.alert.severity", "hour", "day_of_week", "is_weekend", "rule.id", "data.alert.signature_id"]
}
with open('model_config.json', 'w') as f:
    json.dump(config, f)

print("✅ Model exported!")